# 05 — RGAT Inference & LLM+Structural Risk Assessment

This notebook completes the change risk experiment:

1. **Load the trained RGAT model** and compute node embeddings + attention weights
2. **For each PR**, extract a k-hop subgraph around changed nodes with structural context:
   - Attention-weighted dependency pathways
   - Node criticality indicators (PageRank, hub/authority scores, centrality)
   - Cross-module and cross-repo connections
3. **LLM+RGAT risk assessment** — re-prompt GPT-4o with diff **plus** structural context
4. **Compare** LLM-only vs. LLM+RGAT assessments side by side

### Prerequisites
- Trained model in `checkpoints/best_model.pt`
- Preprocessed data in `artifacts/` (config, val_data, model_metadata, node_index)
- PR data from Notebook 04 in `experiment_outputs/`
- GPU recommended (run on Colab)

## 0. Environment Setup

In [20]:
%pip install torch_geometric sentence-transformers igraph leidenalg openai --quiet

In [21]:
# ── Colab Setup ──────────────────────────────────────────────────────
import shutil, zipfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/MSAAI/capstone")
PROJECT_ROOT = Path("/content/rgat_project")
PROJECT_ROOT.mkdir(exist_ok=True)

# Always re-extract source code
zip_src = DRIVE_DIR / "rgat_source.zip"
for pkg in ["rgat", "graph_builder"]:
    old = PROJECT_ROOT / pkg
    if old.exists():
        shutil.rmtree(old)
with zipfile.ZipFile(zip_src, "r") as zf:
    zf.extractall(PROJECT_ROOT)
print("✓ Source code extracted (fresh)")

for d in ["artifacts", "cache", "checkpoints", "experiment_outputs"]:
    (PROJECT_ROOT / d).mkdir(exist_ok=True)

# Copy model artifacts from Drive → project dirs
DRIVE_MODEL = DRIVE_DIR / "model_output"
if DRIVE_MODEL.exists():
    for fname in ["config.pt", "model_metadata.pt", "node_index.json", "training_history.pt", "val_data.pt"]:
        src = DRIVE_MODEL / fname
        dst = PROJECT_ROOT / "artifacts" / fname
        if src.exists() and not dst.exists():
            shutil.copy2(src, dst)
            print(f"  Copied {fname} → artifacts/")
    # best_model.pt → checkpoints/
    bm_src = DRIVE_MODEL / "best_model.pt"
    bm_dst = PROJECT_ROOT / "checkpoints" / "best_model.pt"
    if bm_src.exists() and not bm_dst.exists():
        shutil.copy2(bm_src, bm_dst)
        print("  Copied best_model.pt → checkpoints/")
    print("✓ Model artifacts synced from Drive")

# Copy experiment outputs from Drive (NB04 results)
DRIVE_EXPERIMENT = DRIVE_DIR / "experiment_outputs"
if DRIVE_EXPERIMENT.exists():
    for fname in DRIVE_EXPERIMENT.iterdir():
        dst = PROJECT_ROOT / "experiment_outputs" / fname.name
        if fname.is_file() and not dst.exists():
            shutil.copy2(fname, dst)
            print(f"  Copied {fname.name} → experiment_outputs/")
    print("✓ Experiment outputs synced from Drive")

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Source code extracted (fresh)
✓ Model artifacts synced from Drive
✓ Experiment outputs synced from Drive
PROJECT_ROOT = /content/rgat_project


In [22]:
import os, sys, json, time, warnings
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Optional, Set, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch_geometric.data import HeteroData

from openai import OpenAI

# PROJECT_ROOT set in Colab setup cell
sys.path.insert(0, str(PROJECT_ROOT))

# Force-reload
for mod_name in list(sys.modules):
    if mod_name.startswith("rgat"):
        del sys.modules[mod_name]

from rgat.config import RGATConfig
from rgat.model import HeteroRGATEncoder, RelationDecoder
from rgat.attention import get_attention_weights, attention_to_dataframe

warnings.filterwarnings("ignore", category=FutureWarning)

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
OUTPUT_DIR = PROJECT_ROOT / "experiment_outputs"

# ── API Key (from Drive secrets.json, created in Notebook 04) ────────
secrets_path = DRIVE_DIR / "secrets.json"
assert secrets_path.exists(), (
    f"secrets.json not found at {secrets_path}.\n"
    "Run the one-time setup cell in Notebook 04 first."
)
with open(secrets_path) as _f:
    _secrets = json.load(_f)
OPENAI_API_KEY = _secrets["OPENAI_API_KEY"]
assert OPENAI_API_KEY and not OPENAI_API_KEY.startswith("PASTE"), "Set a real OPENAI_API_KEY in secrets.json"

openai_client = OpenAI(api_key=OPENAI_API_KEY)
OPENAI_MODEL = "gpt-4o"

print("✓ Key loaded from Drive secrets.json")
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

✓ Key loaded from Drive secrets.json
PyTorch 2.10.0+cu128
CUDA available: True


## 1. Load Trained Model & Data

In [23]:
# ── Load config, data, and model metadata ─────────────────────────────
config_dict = torch.load(ARTIFACTS_DIR / "config.pt", weights_only=False)
config = RGATConfig(**config_dict)
config.resolve_device()
device = torch.device(config.device)

# Load graph data
val_data = torch.load(ARTIFACTS_DIR / "val_data.pt", weights_only=False)

# Load model metadata (node/edge types saved during training)
model_meta = torch.load(ARTIFACTS_DIR / "model_metadata.pt", weights_only=False)
all_node_types = model_meta["all_node_types"]
all_edge_types = model_meta["all_edge_types"]

# Load node index and build reverse lookup
with open(ARTIFACTS_DIR / "node_index.json") as f:
    node_index = json.load(f)

reverse_index: Dict[str, Dict[int, str]] = {}
for ntype, mapping in node_index.items():
    reverse_index[ntype] = {v: k for k, v in mapping.items()}

print(f"✓ Config loaded — device={config.device}, hidden_dim={config.hidden_dim}")
print(f"✓ val_data: {len(val_data.node_types)} node types, {len(val_data.edge_types)} edge types")
print(f"✓ Node index: {sum(len(v) for v in node_index.values()):,} nodes")
print(f"✓ Reverse index built")

✓ Config loaded — device=cuda, hidden_dim=128
✓ val_data: 4 node types, 17 edge types
✓ Node index: 120,087 nodes
✓ Reverse index built


In [24]:
# ── Reconstruct and load encoder ─────────────────────────────────────

# Infer num_layers and num_heads from checkpoint to avoid config.pt mismatch
ckpt_path = PROJECT_ROOT / "checkpoints" / "best_model.pt"
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

ckpt_num_layers = max(
    int(k.split(".")[1]) for k in ckpt["encoder_state_dict"] if k.startswith("convs.")
) + 1

# Infer num_heads from att_src shape: [1, sub_heads, head_dim]
# MultiScaleHeteroConv splits num_heads in half (local_heads = num_heads // 2),
# so actual num_heads = sub_heads * 2
att_key = next(k for k in ckpt["encoder_state_dict"] if "local_convs" in k and k.endswith(".att_src"))
sub_heads = ckpt["encoder_state_dict"][att_key].shape[1]
ckpt_num_heads = sub_heads * 2  # local_heads = num_heads // 2

if ckpt_num_layers != config.num_layers:
    print(f"⚠ config.num_layers={config.num_layers} → checkpoint has {ckpt_num_layers}")
    config.num_layers = ckpt_num_layers
if ckpt_num_heads != config.num_heads:
    print(f"⚠ config.num_heads={config.num_heads} → checkpoint has {ckpt_num_heads}")
    config.num_heads = ckpt_num_heads

encoder = HeteroRGATEncoder(
    node_types=all_node_types,
    edge_types=all_edge_types,
    scalar_dims=config.scalar_dims,
    sentence_dim=config.sentence_dim,
    leiden_embed_dim=config.leiden_embed_dim,
    num_leiden_ids=config.num_leiden_ids,
    hidden_dim=config.hidden_dim,
    num_heads=config.num_heads,
    num_layers=config.num_layers,
    dropout=config.dropout,
)

# Identify supervised triplets
supervised_triplets = [
    t for t in val_data.edge_types
    if t[1] in config.supervised_relations
    and hasattr(val_data[t], "edge_label_index")
]

predictor = RelationDecoder(
    hidden_dim=config.hidden_dim,
    supervised_triplets=supervised_triplets,
    decoder_type=config.decoder_type,
)

# Load weights
encoder.load_state_dict(ckpt["encoder_state_dict"])
predictor.load_state_dict(ckpt["predictor_state_dict"])

encoder = encoder.to(device)
predictor = predictor.to(device)
val_data = val_data.to(device)

encoder.eval()
predictor.eval()

print(f"✓ Model loaded from {ckpt_path}")
print(f"  num_layers={config.num_layers}, num_heads={config.num_heads}, hidden_dim={config.hidden_dim}")
print(f"  Encoder params: {sum(p.numel() for p in encoder.parameters()):,}")
print(f"  Decoder params: {sum(p.numel() for p in predictor.parameters()):,}")

⚠ config.num_layers=2 → checkpoint has 3
⚠ config.num_heads=4 → checkpoint has 8
✓ Model loaded from /content/rgat_project/checkpoints/best_model.pt
  num_layers=3, num_heads=8, hidden_dim=128
  Encoder params: 4,236,003
  Decoder params: 512


## 2. Compute Embeddings & Attention Weights

Run a single forward pass to get all node embeddings and per-layer attention.

In [25]:
# ── Forward pass with attention ──────────────────────────────────────
with torch.no_grad():
    z_dict, all_attn = encoder(val_data, return_attention_weights=True)

print("Embedding shapes:")
for ntype, z in z_dict.items():
    print(f"  {ntype}: {z.shape}")

print(f"\nAttention collected from {len(all_attn)} layers")
for layer_idx, layer_attn in all_attn.items():
    print(f"  Layer {layer_idx}: {len(layer_attn)} edge type entries")

# ── Precompute global attention statistics (once, not per-PR) ────────
_all_attn_values = []
for layer_idx, layer_attn in all_attn.items():
    for key, alpha in layer_attn.items():
        parts = key.split("|")
        if len(parts) != 4:
            continue
        scale, src_type, rel, dst_type = parts
        et = (src_type, rel, dst_type)
        if not hasattr(val_data[et], "edge_index"):
            continue
        n_edges = min(val_data[et].edge_index.size(1), alpha.size(0))
        if alpha.dim() == 2:
            mean_alpha = alpha[:n_edges].mean(dim=1).cpu()
        else:
            mean_alpha = alpha[:n_edges].cpu()
        _all_attn_values.extend(mean_alpha.tolist())

global_attn_arr = np.array(_all_attn_values)
global_attn_stats = {
    "mean": float(np.mean(global_attn_arr)),
    "std": float(np.std(global_attn_arr)),
    "p50": float(np.percentile(global_attn_arr, 50)),
    "p90": float(np.percentile(global_attn_arr, 90)),
    "p95": float(np.percentile(global_attn_arr, 95)),
    "p99": float(np.percentile(global_attn_arr, 99)),
    "max": float(np.max(global_attn_arr)),
}
del _all_attn_values  # free memory

print(f"\nGlobal attention stats precomputed ({len(global_attn_arr):,} values)")
print(f"  median={global_attn_stats['p50']:.4f}, p95={global_attn_stats['p95']:.4f}, p99={global_attn_stats['p99']:.4f}")

Embedding shapes:
  class: torch.Size([26478, 128])
  function: torch.Size([81479, 128])
  module: torch.Size([12117, 128])
  repo: torch.Size([13, 128])

Attention collected from 3 layers
  Layer 0: 34 edge type entries
  Layer 1: 34 edge type entries
  Layer 2: 34 edge type entries

Global attention stats precomputed (2,543,874 values)
  median=0.2952, p95=1.0000, p99=1.0000


## 3. Load PR Data from Notebook 04

In [26]:
# ── Load PR data and LLM-only results ────────────────────────────────
with open(OUTPUT_DIR / "04_pr_data_and_llm_baseline.json") as f:
    experiment_data = json.load(f)

all_prs = experiment_data["prs"]
llm_only_results = experiment_data["llm_only_results"]

# Build lookup for LLM-only results
llm_only_by_pr = {r["pr_id"]: r["assessment"] for r in llm_only_results}

# Load compact node mapping
with open(OUTPUT_DIR / "04_pr_node_mapping.json") as f:
    pr_node_mapping = json.load(f)

print(f"Loaded {len(all_prs)} PRs")
print(f"LLM-only results: {len(llm_only_results)}")
print(f"Node mappings: {len(pr_node_mapping)}")

Loaded 55 PRs
LLM-only results: 55
Node mappings: 55


## 4. Structural Context Extraction

For each PR, extract a k-hop subgraph around the changed nodes and compute:
- Attention-weighted incoming/outgoing dependency scores
- Node structural features (PageRank, hub/authority, degree)
- Dependency paths and cross-module connections
- Embedding similarity to highly-central nodes

In [27]:
def resolve_node_indices(node_ids: List[str]) -> Dict[str, List[int]]:
    """Convert string node IDs to (node_type, integer_index) pairs."""
    indices_by_type: Dict[str, List[int]] = defaultdict(list)
    for nid in node_ids:
        if nid.startswith("mod::"):
            ntype = "module"
        elif nid.startswith("class::"):
            ntype = "class"
        elif nid.startswith("func::"):
            ntype = "function"
        elif nid.startswith("repo::"):
            ntype = "repo"
        else:
            continue
        idx = node_index.get(ntype, {}).get(nid)
        if idx is not None:
            indices_by_type[ntype].append(idx)
    return dict(indices_by_type)


def get_node_scalar_features(ntype: str, idx: int) -> Dict[str, float]:
    """Extract raw scalar features for a node."""
    from rgat.config import REQUIRED_SCALAR_FEATURES
    feature_names = REQUIRED_SCALAR_FEATURES.get(ntype, [])
    features = {}
    x_scalar = val_data[ntype].x_scalar[idx].cpu()
    for i, name in enumerate(feature_names):
        if i < len(x_scalar):
            features[name] = round(float(x_scalar[i]), 4)
    return features


def get_k_hop_neighbors(
    seed_indices: Dict[str, List[int]],
    k: int = 2,
    max_neighbors_per_hop: int = 25,
) -> Dict[str, Set[int]]:
    """Find k-hop neighbors via GPU-vectorized torch.isin()."""
    visited: Dict[str, Set[int]] = defaultdict(set)
    for ntype, idxs in seed_indices.items():
        visited[ntype].update(idxs)

    # Build GPU tensors for frontier sets
    frontier_tensors: Dict[str, torch.Tensor] = {
        ntype: torch.tensor(idxs, dtype=torch.long, device=device)
        for ntype, idxs in seed_indices.items()
    }

    for hop in range(k):
        next_frontier: Dict[str, Set[int]] = defaultdict(set)

        for et in all_edge_types:
            src_type, rel, dst_type = et
            if not hasattr(val_data[et], "edge_index"):
                continue
            ei = val_data[et].edge_index  # stays on GPU

            # Forward: frontier nodes as source → find destinations
            if src_type in frontier_tensors:
                mask = torch.isin(ei[0], frontier_tensors[src_type])
                if mask.any():
                    dst_idxs = ei[1, mask].unique()
                    # Filter already visited
                    if visited[dst_type]:
                        visited_t = torch.tensor(list(visited[dst_type]), dtype=torch.long, device=device)
                        new_mask = ~torch.isin(dst_idxs, visited_t)
                        dst_idxs = dst_idxs[new_mask]
                    dst_idxs = dst_idxs[:max_neighbors_per_hop].cpu().tolist()
                    next_frontier[dst_type].update(dst_idxs)

            # Backward: frontier nodes as destination → find sources
            if dst_type in frontier_tensors:
                mask = torch.isin(ei[1], frontier_tensors[dst_type])
                if mask.any():
                    src_idxs = ei[0, mask].unique()
                    if visited[src_type]:
                        visited_t = torch.tensor(list(visited[src_type]), dtype=torch.long, device=device)
                        new_mask = ~torch.isin(src_idxs, visited_t)
                        src_idxs = src_idxs[new_mask]
                    src_idxs = src_idxs[:max_neighbors_per_hop].cpu().tolist()
                    next_frontier[src_type].update(src_idxs)

        # Update visited and rebuild frontier tensors
        frontier_tensors = {}
        for ntype, idxs in next_frontier.items():
            visited[ntype].update(idxs)
            if idxs:
                frontier_tensors[ntype] = torch.tensor(list(idxs), dtype=torch.long, device=device)

    return dict(visited)


print("✓ Subgraph extraction functions ready (GPU-accelerated)")

✓ Subgraph extraction functions ready (GPU-accelerated)


In [28]:
def extract_attention_context(
    seed_indices: Dict[str, List[int]],
    subgraph_nodes: Dict[str, Set[int]],
    top_k_edges: int = 25,
) -> Tuple[List[Dict], Dict]:
    """Extract highest-attention edges via GPU-vectorized torch.isin()."""
    # Pre-build GPU tensors for subgraph and seed membership
    subgraph_tensors: Dict[str, torch.Tensor] = {
        ntype: torch.tensor(list(idxs), dtype=torch.long, device=device)
        for ntype, idxs in subgraph_nodes.items() if idxs
    }
    seed_tensors: Dict[str, torch.Tensor] = {
        ntype: torch.tensor(idxs, dtype=torch.long, device=device)
        for ntype, idxs in seed_indices.items()
    }

    # Collect candidates: (attention_value, metadata_dict)
    candidates = []

    for layer_idx, layer_attn in all_attn.items():
        for key, alpha in layer_attn.items():
            parts = key.split("|")
            if len(parts) != 4:
                continue
            scale, src_type, rel, dst_type = parts

            et = (src_type, rel, dst_type)
            if not hasattr(val_data[et], "edge_index"):
                continue

            # Skip if no subgraph nodes on either side
            if src_type not in subgraph_tensors and dst_type not in subgraph_tensors:
                continue

            ei = val_data[et].edge_index  # on GPU
            n_edges = min(ei.size(1), alpha.size(0))

            # Mean attention across heads (stay on GPU)
            if alpha.dim() == 2:
                mean_alpha = alpha[:n_edges].mean(dim=1)
            else:
                mean_alpha = alpha[:n_edges]

            # Vectorized membership check on GPU
            src_nodes = ei[0, :n_edges]
            dst_nodes = ei[1, :n_edges]

            src_mask = torch.isin(src_nodes, subgraph_tensors[src_type]) if src_type in subgraph_tensors else torch.zeros(n_edges, dtype=torch.bool, device=device)
            dst_mask = torch.isin(dst_nodes, subgraph_tensors[dst_type]) if dst_type in subgraph_tensors else torch.zeros(n_edges, dtype=torch.bool, device=device)
            edge_mask = src_mask | dst_mask

            if not edge_mask.any():
                continue

            # Get matching indices and their attention values
            match_idx = edge_mask.nonzero(as_tuple=True)[0]
            match_attn = mean_alpha[match_idx]
            match_src = src_nodes[match_idx]
            match_dst = dst_nodes[match_idx]

            # Seed membership for matched edges
            src_is_seed = torch.isin(match_src, seed_tensors[src_type]) if src_type in seed_tensors else torch.zeros(match_idx.size(0), dtype=torch.bool, device=device)
            dst_is_seed = torch.isin(match_dst, seed_tensors[dst_type]) if dst_type in seed_tensors else torch.zeros(match_idx.size(0), dtype=torch.bool, device=device)
            is_seed = src_is_seed | dst_is_seed

            # Transfer to CPU in batch
            attn_cpu = match_attn.cpu().tolist()
            src_cpu = match_src.cpu().tolist()
            dst_cpu = match_dst.cpu().tolist()
            seed_cpu = is_seed.cpu().tolist()

            for j in range(len(attn_cpu)):
                candidates.append((attn_cpu[j], {
                    "layer": layer_idx,
                    "scale": scale,
                    "relation": rel,
                    "src_type": src_type,
                    "dst_type": dst_type,
                    "src_idx": src_cpu[j],
                    "dst_idx": dst_cpu[j],
                    "src_id": reverse_index.get(src_type, {}).get(src_cpu[j], f"{src_type}:{src_cpu[j]}"),
                    "dst_id": reverse_index.get(dst_type, {}).get(dst_cpu[j], f"{dst_type}:{dst_cpu[j]}"),
                    "attention": attn_cpu[j],
                    "is_seed_edge": bool(seed_cpu[j]),
                }))

    # Sort by attention (descending) and keep top-k
    candidates.sort(key=lambda x: -x[0])
    results = [c[1] for c in candidates[:top_k_edges]]

    # Assign percentile ranks to the top-k only
    for r in results:
        r["percentile"] = float(np.mean(global_attn_arr <= r["attention"]) * 100)

    return results, global_attn_stats


print("✓ Attention context extractor ready (GPU-accelerated)")

✓ Attention context extractor ready (GPU-accelerated)


In [29]:
def build_structural_context(pr_data: Dict, pr_nodes_entry: Dict) -> Dict:
    """Build complete structural context for a PR.

    Returns a dict with all structural intelligence for the LLM prompt.
    """
    node_ids = pr_nodes_entry["node_ids"]
    if not node_ids:
        return {"error": "No matched nodes in graph"}

    # Resolve to integer indices
    seed_indices = resolve_node_indices(node_ids)
    if not seed_indices:
        return {"error": "No nodes found in current graph"}

    # ── 1. Node-level structural features ─────────────────────────
    node_profiles = []
    for ntype, idxs in seed_indices.items():
        for idx in idxs:
            nid = reverse_index.get(ntype, {}).get(idx, f"{ntype}:{idx}")
            features = get_node_scalar_features(ntype, idx)
            emb = z_dict[ntype][idx].cpu()
            node_profiles.append({
                "node_id": nid,
                "node_type": ntype,
                "pagerank": features.get("pagerank", 0),
                "hub_score": features.get("hub_score", 0),
                "authority_score": features.get("authority_score", 0),
                "in_degree": features.get("in_degree", 0),
                "out_degree": features.get("out_degree", 0),
                "key_features": {
                    k: v for k, v in features.items()
                    if k not in ("pagerank", "hub_score", "authority_score",
                                 "in_degree", "out_degree")
                },
            })

    # ── 2. K-hop subgraph ─────────────────────────────────────────
    subgraph_nodes = get_k_hop_neighbors(seed_indices, k=2, max_neighbors_per_hop=25)
    subgraph_size = sum(len(v) for v in subgraph_nodes.values())

    # ── 3. Attention-weighted dependency pathways ─────────────────
    top_attention_edges, attn_stats = extract_attention_context(
        seed_indices, subgraph_nodes, top_k_edges=25,
    )

    # ── 4. Cross-module / cross-repo connections ──────────────────
    cross_module = []
    cross_repo = []
    for edge in top_attention_edges:
        src_repo = edge["src_id"].split("::")[1] if "::" in edge["src_id"] else ""
        dst_repo = edge["dst_id"].split("::")[1] if "::" in edge["dst_id"] else ""

        if src_repo and dst_repo:
            if src_repo != dst_repo:
                cross_repo.append(edge)
            else:
                src_mod = edge["src_id"].split("::")[2] if len(edge["src_id"].split("::")) > 2 else ""
                dst_mod = edge["dst_id"].split("::")[2] if len(edge["dst_id"].split("::")) > 2 else ""
                if src_mod and dst_mod and src_mod.split(".")[1:2] != dst_mod.split(".")[1:2]:
                    cross_module.append(edge)

    # ── 5. Embedding-based similarity (cosine) to important neighbors ─
    embedding_similarities = []
    for ntype, seed_idxs in seed_indices.items():
        neighbor_idxs = subgraph_nodes.get(ntype, set()) - set(seed_idxs)
        if not neighbor_idxs:
            continue
        for s_idx in seed_idxs:
            s_emb = z_dict[ntype][s_idx]
            for n_idx in list(neighbor_idxs)[:20]:
                n_emb = z_dict[ntype][n_idx]
                sim = float(F.cosine_similarity(s_emb.unsqueeze(0), n_emb.unsqueeze(0)))
                if sim > 0.7:
                    n_id = reverse_index.get(ntype, {}).get(n_idx, f"{ntype}:{n_idx}")
                    s_id = reverse_index.get(ntype, {}).get(s_idx, f"{ntype}:{s_idx}")
                    embedding_similarities.append({
                        "source": s_id,
                        "neighbor": n_id,
                        "cosine_similarity": round(sim, 4),
                    })

    embedding_similarities.sort(key=lambda x: -x["cosine_similarity"])

    return {
        "node_profiles": node_profiles,
        "subgraph_size": subgraph_size,
        "top_attention_edges": top_attention_edges,
        "attn_stats": attn_stats,
        "cross_module_edges": cross_module[:10],
        "cross_repo_edges": cross_repo[:10],
        "high_similarity_neighbors": embedding_similarities[:15],
        "num_seed_nodes": sum(len(v) for v in seed_indices.values()),
    }


print("✓ Structural context builder ready")

✓ Structural context builder ready


## 5. Extract Structural Context for All PRs

In [30]:
# ── Build structural context for each PR ─────────────────────────────
import pickle

ctx_cache_path = OUTPUT_DIR / "05_structural_contexts.pkl"

if ctx_cache_path.exists():
    with open(ctx_cache_path, "rb") as f:
        structural_contexts = pickle.load(f)
    print(f"✓ Loaded cached structural contexts ({len(structural_contexts)} PRs)")
else:
    structural_contexts: Dict[str, Dict] = {}

    for i, pr_nodes_entry in enumerate(pr_node_mapping):
        pr_id = pr_nodes_entry["pr_id"]
        pr_data = all_prs[i]  # aligned by index

        print(f"[{i+1}/{len(pr_node_mapping)}] Extracting context for {pr_id}...")

        ctx = build_structural_context(pr_data, pr_nodes_entry)

        if "error" in ctx:
            print(f"  ⚠ {ctx['error']}")
        else:
            print(
                f"  ✓ {ctx['num_seed_nodes']} seed nodes, "
                f"{ctx['subgraph_size']} subgraph nodes, "
                f"{len(ctx['top_attention_edges'])} attn edges, "
                f"{len(ctx['cross_repo_edges'])} cross-repo"
            )

        structural_contexts[pr_id] = ctx

    # Cache to disk so kernel restarts don't lose this
    with open(ctx_cache_path, "wb") as f:
        pickle.dump(structural_contexts, f)

    # Also copy to Drive
    DRIVE_EXPERIMENT.mkdir(exist_ok=True)
    shutil.copy2(ctx_cache_path, DRIVE_EXPERIMENT / ctx_cache_path.name)

    print(f"\n✓ Extracted & cached structural context for {len(structural_contexts)} PRs")

✓ Loaded cached structural contexts (55 PRs)


## 6. LLM+RGAT Risk Assessment

Re-prompt GPT-4o with the same diff **plus** the structural context from the RGAT model.

In [39]:
RISK_SYSTEM_PROMPT_AUGMENTED = """You are an expert software architect specialising in change risk assessment for large Python ecosystems.

You will be given:
1. A pull request diff from an open-source Django ecosystem project
2. **Supplemental structural context** from a trained Relational Graph Attention Network (RGAT) that has learned dependency patterns across 13 interconnected Django ecosystem repositories

**Important: Your primary basis for risk assessment is the code diff itself.** The structural context is supplemental — it provides visibility into *outside factors* you cannot see from the diff alone, such as:
- Which downstream components depend on the changed code
- Whether the changed code sits at a critical junction in the dependency graph
- Cross-module and cross-repo relationships that the diff doesn't reveal

**How to use the structural context:**
- The RGAT attention rankings show dependency relationships near the changed code, ranked by the model's learned importance. Use these to understand what *else* connects to the changed code — not to inflate risk simply because high-attention edges exist nearby.
- An edge marked ★ CHANGED directly involves a modified node. Unmarked edges are neighbours — they provide context about the surrounding architecture but should not independently drive up the risk score.
- High percentile rankings mean the model considers that relationship important *in general*. This only matters for risk if the change actually affects that relationship (e.g., modifying a function's signature when it has high-attention CALLS edges).
- Node centrality (PageRank, degree) tells you whether a changed component is widely depended upon. A high-centrality node being modified is more concerning than a peripheral one — but only if the change is substantive (not just formatting, docs, or tests).

**When structural context SHOULD increase risk:**
- A ★ CHANGED edge has high attention AND the diff modifies the interface that edge depends on (e.g., changing a function signature, altering return types, renaming or removing a public method). This is evidence of real propagation risk — raise probability.
- A changed node has high PageRank or high in-degree AND the diff makes a substantive change (not just docs/tests/formatting). A widely-depended-upon component being meaningfully modified deserves higher severity.
- Cross-repo ★ CHANGED edges indicate the change touches a component that other repositories depend on. If the diff alters the contract those repos rely on, this is a genuine cross-project risk.
- Multiple high-attention ★ CHANGED edges converging on the same changed node suggest it is a critical hub — the structural context is revealing risk that the diff alone understates.

**When structural context should NOT increase risk:**
- The neighbourhood contains high-percentile edges that don't involve the actual code change (no ★ CHANGED marker).
- The change is CI/config/docs/test-only — surrounding graph structure doesn't make these high risk.
- Edges are merely in the 2-hop neighbourhood but unrelated to the semantic change.

Risk = **Severity × Probability**

**Severity (1–10):** How impactful could a defect in this change be?
- Base this primarily on what the diff actually changes (API signatures, core logic, data models vs. cosmetic/docs/tests)
- Use structural context to check: is the changed component widely depended upon? (high in-degree, high PageRank)
- Cross-repo dependencies amplify severity only if the change modifies the interface those dependencies rely on

**Probability (1–10):** How likely is this change to introduce or propagate issues?
- Base this primarily on the complexity and nature of the code change itself
- Use structural context to identify propagation pathways: if the diff modifies a function that has high-attention CALLS edges from many callers (★ CHANGED), the probability of downstream breakage increases
- A simple, well-scoped change has low probability even if it touches a high-centrality node

**Respond with valid JSON only**, matching this exact schema:
```json
{
  "severity": <int 1-10>,
  "severity_reasoning": "<2-3 sentences explaining severity based on the diff, noting any relevant structural context>",
  "probability": <int 1-10>,
  "probability_reasoning": "<2-3 sentences explaining probability based on the diff, noting any relevant propagation pathways>",
  "risk_score": <int: severity * probability>,
  "risk_level": "<low|medium|high|critical>",
  "key_risks": ["<risk 1>", "<risk 2>", ...],
  "affected_areas": ["<area 1>", "<area 2>", ...],
  "structural_insights": ["<insight from the structural context that adds to understanding beyond the diff>", ...],
  "summary": "<1 paragraph: what does the diff change, and what does the structural context reveal about its broader impact?>"
}
```

Risk level thresholds: low (1-15), medium (16-35), high (36-64), critical (65-100).
"""


def format_structural_context(ctx: Dict) -> str:
    """Format structural context with attention-ranked relationships first."""
    if "error" in ctx:
        return f"(Structural context unavailable: {ctx['error']})"

    sections = []
    attn_stats = ctx.get("attn_stats", {})

    # ══════════════════════════════════════════════════════════════
    # SECTION 1 (PRIMARY): Attention-ranked relationships
    # ══════════════════════════════════════════════════════════════
    if ctx["top_attention_edges"]:
        sections.append("## RGAT Attention-Ranked Relationships (Supplemental Context)")
        sections.append(
            "Dependencies near the changed code, ranked by the model's learned "
            "attention scores. These reveal the surrounding dependency architecture "
            "— use them to understand what connects to the changed code, not as a "
            "direct risk signal."
        )
        if attn_stats:
            sections.append(
                f"_Global attention baseline: "
                f"median={attn_stats['p50']:.4f}, "
                f"90th pctl={attn_stats['p90']:.4f}, "
                f"95th pctl={attn_stats['p95']:.4f}, "
                f"99th pctl={attn_stats['p99']:.4f}_"
            )
        sections.append("")

        for rank, edge in enumerate(ctx["top_attention_edges"], 1):
            pctl = edge.get("percentile", 0)
            seed_marker = " ★ CHANGED" if edge["is_seed_edge"] else ""

            # Percentile label
            if pctl >= 99:
                pctl_label = f"TOP 1% (p{pctl:.0f})"
            elif pctl >= 95:
                pctl_label = f"TOP 5% (p{pctl:.0f})"
            elif pctl >= 90:
                pctl_label = f"TOP 10% (p{pctl:.0f})"
            else:
                pctl_label = f"p{pctl:.0f}"

            # Flag cross-repo
            src_repo = edge["src_id"].split("::")[1] if "::" in edge["src_id"] else ""
            dst_repo = edge["dst_id"].split("::")[1] if "::" in edge["dst_id"] else ""
            cross_flag = " CROSS-REPO" if (src_repo and dst_repo and src_repo != dst_repo) else ""

            sections.append(
                f"{rank}. **{edge['relation']}** | attn={edge['attention']:.4f} | {pctl_label}{cross_flag}{seed_marker}\n"
                f"   {edge['src_id']} → {edge['dst_id']}"
            )

    # ══════════════════════════════════════════════════════════════
    # SECTION 2: Changed node summary (compact)
    # ══════════════════════════════════════════════════════════════
    if ctx["node_profiles"]:
        sections.append("\n## Changed Node Profiles")
        for np_ in ctx["node_profiles"]:
            notable = {k: v for k, v in np_["key_features"].items()
                       if isinstance(v, (int, float)) and v > 0
                       and k in ("cyclomatic_complexity", "num_methods", "inheritance_depth",
                                 "num_calls_made", "import_fan_out", "import_fan_in",
                                 "num_bases", "is_abstract", "loc")}
            feat_str = f"  [{', '.join(f'{k}={v}' for k, v in notable.items())}]" if notable else ""
            sections.append(
                f"- **{np_['node_id']}** — "
                f"PR={np_['pagerank']:.4f}, in={np_['in_degree']}, out={np_['out_degree']}"
                f"{feat_str}"
            )

    # ══════════════════════════════════════════════════════════════
    # SECTION 3: Embedding similarities (compact)
    # ══════════════════════════════════════════════════════════════
    if ctx["high_similarity_neighbors"]:
        sections.append("\n## Structurally Similar Components")
        for sim in ctx["high_similarity_neighbors"][:8]:
            sections.append(
                f"- {sim['source']} ↔ {sim['neighbor']} (cos={sim['cosine_similarity']:.3f})"
            )

    # Subgraph size as a footnote
    sections.append(
        f"\n_2-hop dependency neighbourhood: {ctx['subgraph_size']} nodes_"
    )

    return "\n".join(sections)


def build_augmented_prompt(pr_data: Dict, structural_text: str) -> str:
    """Build the user prompt with both diff and structural context."""
    entities = pr_data["entities"]
    changed_files_str = "\n".join(f"  - {f}" for f in entities["changed_files"][:20])
    if len(entities["changed_files"]) > 20:
        changed_files_str += f"\n  ... and {len(entities['changed_files']) - 20} more"

    prompt = f"""## Pull Request: {pr_data['pr_title']}
**Repository:** {pr_data['github_path']}
**PR:** #{pr_data['pr_number']} ({pr_data['pr_url']})
**Files changed:** {pr_data['changed_files_count']} (+{pr_data['additions']}/−{pr_data['deletions']})
**Merged:** {pr_data['merged_at']}

### PR Description
{pr_data['pr_body'][:1000] if pr_data['pr_body'] else '(no description)'}

### Changed Files
{changed_files_str}

---

### Diff (Primary Evidence)
```diff
{pr_data['diff_text']}
```

---

## Supplemental: Structural Context from RGAT Model
The following structural intelligence comes from a trained graph neural network.
Use it to understand the broader dependency context around the changed code — what
depends on it, what it depends on, and how central it is in the ecosystem. This
context supplements your diff analysis; it should not override it.

{structural_text}

Based on the diff above and the supplemental structural context, provide your risk assessment as JSON.
"""
    return prompt


print("✓ Augmented prompt builder ready")

✓ Augmented prompt builder ready


In [40]:
def call_openai_risk_assessment(
    system_prompt: str,
    user_prompt: str,
    model: str = OPENAI_MODEL,
    max_retries: int = 3,
) -> Optional[Dict]:
    """Call OpenAI API and parse structured JSON response."""
    for attempt in range(max_retries):
        try:
            response = openai_client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.2,
                response_format={"type": "json_object"},
                max_tokens=1200,
            )

            content = response.choices[0].message.content
            result = json.loads(content)

            required = ["severity", "probability", "risk_score", "risk_level", "summary"]
            if all(k in result for k in required):
                result["_model"] = model
                result["_tokens"] = {
                    "prompt": response.usage.prompt_tokens,
                    "completion": response.usage.completion_tokens,
                }
                return result
            else:
                missing = [k for k in required if k not in result]
                print(f"  ⚠ Missing keys: {missing}. Retrying...")

        except json.JSONDecodeError as e:
            print(f"  ⚠ JSON parse error (attempt {attempt+1}): {e}")
        except Exception as e:
            print(f"  ⚠ API error (attempt {attempt+1}): {e}")
            time.sleep(2 ** attempt)

    return None


print("✓ OpenAI caller ready")

✓ OpenAI caller ready


In [41]:
# ── Run LLM+RGAT risk assessment ─────────────────────────────────────
augmented_results = []

for i, pr_data in enumerate(all_prs):
    pr_id = f"{pr_data['repo_short']}#{pr_data['pr_number']}"
    print(f"[{i+1}/{len(all_prs)}] Augmented assessment for {pr_id}...")

    ctx = structural_contexts.get(pr_id, {"error": "Not computed"})
    structural_text = format_structural_context(ctx)
    user_prompt = build_augmented_prompt(pr_data, structural_text)

    assessment = call_openai_risk_assessment(
        RISK_SYSTEM_PROMPT_AUGMENTED, user_prompt,
    )

    if assessment:
        print(
            f"  → Risk: {assessment['risk_score']} ({assessment['risk_level']}) "
            f"[S={assessment['severity']} × P={assessment['probability']}]"
        )
    else:
        print(f"  ✗ Failed")
        assessment = {"error": "API call failed"}

    augmented_results.append({
        "pr_id": pr_id,
        "repo_short": pr_data["repo_short"],
        "pr_number": pr_data["pr_number"],
        "pr_title": pr_data["pr_title"],
        "assessment": assessment,
        "structural_context_summary": {
            "num_seed_nodes": ctx.get("num_seed_nodes", 0),
            "subgraph_size": ctx.get("subgraph_size", 0),
            "num_attention_edges": len(ctx.get("top_attention_edges", [])),
            "num_cross_repo": len(ctx.get("cross_repo_edges", [])),
        },
    })

    time.sleep(0.5)

print(f"\n✓ Completed {len(augmented_results)} augmented assessments")

[1/55] Augmented assessment for django#21050...
  → Risk: 2 (low) [S=2 × P=1]
[2/55] Augmented assessment for django#21046...
  → Risk: 12 (low) [S=4 × P=3]
[3/55] Augmented assessment for django#20889...
  → Risk: 12 (low) [S=4 × P=3]
[4/55] Augmented assessment for django#20027...
  → Risk: 12 (low) [S=4 × P=3]
[5/55] Augmented assessment for django#21035...
  → Risk: 12 (low) [S=4 × P=3]
[6/55] Augmented assessment for drf#9902...
  → Risk: 15 (low) [S=5 × P=3]
[7/55] Augmented assessment for drf#9929...
  → Risk: 12 (low) [S=4 × P=3]
[8/55] Augmented assessment for drf#9931...
  → Risk: 2 (low) [S=2 × P=1]
[9/55] Augmented assessment for drf#9928...
  → Risk: 6 (low) [S=3 × P=2]
[10/55] Augmented assessment for drf#9735...
  → Risk: 12 (low) [S=4 × P=3]
[11/55] Augmented assessment for wagtail#14017...
  → Risk: 24 (medium) [S=6 × P=4]
[12/55] Augmented assessment for wagtail#14034...
  → Risk: 30 (medium) [S=6 × P=5]
[13/55] Augmented assessment for wagtail#13930...
  → Risk: 30 (

## 7. Side-by-Side Comparison

In [42]:
# ── Build comparison table ────────────────────────────────────────────
comparison_rows = []

for aug in augmented_results:
    pr_id = aug["pr_id"]
    llm_only = llm_only_by_pr.get(pr_id, {})
    aug_assess = aug["assessment"]

    if "error" in llm_only or "error" in aug_assess:
        continue

    comparison_rows.append({
        "PR": pr_id,
        "title": aug["pr_title"][:35],
        "LLM_severity": llm_only.get("severity", "-"),
        "AUG_severity": aug_assess.get("severity", "-"),
        "sev_delta": aug_assess.get("severity", 0) - llm_only.get("severity", 0),
        "LLM_probability": llm_only.get("probability", "-"),
        "AUG_probability": aug_assess.get("probability", "-"),
        "prob_delta": aug_assess.get("probability", 0) - llm_only.get("probability", 0),
        "LLM_risk": llm_only.get("risk_score", "-"),
        "AUG_risk": aug_assess.get("risk_score", "-"),
        "risk_delta": aug_assess.get("risk_score", 0) - llm_only.get("risk_score", 0),
        "LLM_level": llm_only.get("risk_level", "-"),
        "AUG_level": aug_assess.get("risk_level", "-"),
    })

df_comparison = pd.DataFrame(comparison_rows)
print("\n" + "="*80)
print("SIDE-BY-SIDE COMPARISON: LLM-Only vs. LLM+RGAT")
print("="*80)
display(df_comparison)

if len(df_comparison) > 0:
    print(f"\n--- Summary Statistics ---")
    print(f"Mean severity delta:    {df_comparison['sev_delta'].mean():+.2f}")
    print(f"Mean probability delta: {df_comparison['prob_delta'].mean():+.2f}")
    print(f"Mean risk score delta:  {df_comparison['risk_delta'].mean():+.2f}")
    print(f"Risk level changes:     {(df_comparison['LLM_level'] != df_comparison['AUG_level']).sum()} / {len(df_comparison)}")


SIDE-BY-SIDE COMPARISON: LLM-Only vs. LLM+RGAT


,PR,title,LLM_severity,AUG_severity,sev_delta,LLM_probability,AUG_probability,prob_delta,LLM_risk,AUG_risk,risk_delta,LLM_level,AUG_level
0,django#21050,Refs #36949 -- Removed hardcoded pk,2,2,0,1,1,0,2,2,0,low,low
1,django#21046,Fixed #37016 -- Avoided propagating,6,4,-2,4,3,-1,24,12,-12,medium,low
2,django#20889,Fixed #36973 -- Made fields.E348 de,6,4,-2,4,3,-1,24,12,-12,medium,low
3,django#20027,Fixed #20024 -- Fixed handling of _,7,4,-3,5,3,-2,35,12,-23,medium,low
4,django#21035,Fixed #36949 -- Improved RelatedFie,5,4,-1,4,3,-1,20,12,-8,medium,low
5,drf#9902,Fix partial form data updates invol,6,5,-1,4,3,-1,24,15,-9,medium,low
6,drf#9929,Include `choices` param for non-edi,6,4,-2,4,3,-1,24,12,-12,medium,low
7,drf#9931,Prepare bug fix release 3.17.1,3,2,-1,2,1,-1,6,2,-4,low,low
8,drf#9928,Fix `HTMLFormRenderer` with empty `,6,3,-3,3,2,-1,18,6,-12,medium,low
9,drf#9735,Preserve ordering in `MultipleChoic,7,4,-3,6,3,-3,42,12,-30,high,low



--- Summary Statistics ---
Mean severity delta:    -1.62
Mean probability delta: -1.00
Mean risk score delta:  -10.02
Risk level changes:     32 / 55


In [43]:
# ── Detailed per-PR comparison with reasoning ────────────────────────
print("\n" + "="*80)
print("DETAILED COMPARISON (per PR)")
print("="*80)

for aug in augmented_results:
    pr_id = aug["pr_id"]
    llm_only = llm_only_by_pr.get(pr_id, {})
    aug_assess = aug["assessment"]

    if "error" in llm_only or "error" in aug_assess:
        continue

    print(f"\n{'─'*60}")
    print(f"PR: {pr_id} — {aug['pr_title'][:60]}")
    print(f"{'─'*60}")

    print(f"\n  LLM-Only  → Risk={llm_only['risk_score']} ({llm_only['risk_level']}) "
          f"[S={llm_only['severity']}, P={llm_only['probability']}]")
    print(f"    Severity: {llm_only.get('severity_reasoning', 'N/A')}")
    print(f"    Probability: {llm_only.get('probability_reasoning', 'N/A')}")
    print(f"    Key risks: {llm_only.get('key_risks', [])}")

    print(f"\n  LLM+RGAT  → Risk={aug_assess['risk_score']} ({aug_assess['risk_level']}) "
          f"[S={aug_assess['severity']}, P={aug_assess['probability']}]")
    print(f"    Severity: {aug_assess.get('severity_reasoning', 'N/A')}")
    print(f"    Probability: {aug_assess.get('probability_reasoning', 'N/A')}")
    print(f"    Key risks: {aug_assess.get('key_risks', [])}")
    if "structural_insights" in aug_assess:
        print(f"    Structural insights: {aug_assess['structural_insights']}")

    delta_risk = aug_assess.get('risk_score', 0) - llm_only.get('risk_score', 0)
    direction = "↑ HIGHER" if delta_risk > 0 else ("↓ LOWER" if delta_risk < 0 else "= SAME")
    print(f"\n  Delta: {direction} (risk: {delta_risk:+d})")


DETAILED COMPARISON (per PR)

────────────────────────────────────────────────────────────
PR: django#21050 — Refs #36949 -- Removed hardcoded pks in modeladmin tests.
────────────────────────────────────────────────────────────

  LLM-Only  → Risk=2 (low) [S=2, P=1]
    Severity: The change is limited to test code within the Django test suite, specifically affecting the modeladmin tests. It does not impact core modules, public APIs, or critical paths of the Django framework.
    Probability: The change is a minor refactor in test code, replacing hardcoded primary keys with dynamic values. It is unlikely to cause issues as it does not affect production code or alter any function signatures or class hierarchies.
    Key risks: ['Potential for test failures if dynamic PKs are not correctly set.']

  LLM+RGAT  → Risk=2 (low) [S=2, P=1]
    Severity: The change is limited to test code, specifically modifying how primary keys are referenced in assertions. This does not impact the core logi

## 8. Save All Experiment Results

In [44]:
# ── Save complete experiment results ─────────────────────────────────
full_results = {
    "config": {
        "openai_model": OPENAI_MODEL,
        "k_hop": 2,
        "max_neighbors_per_hop": 25,
        "top_k_attention_edges": 25,
        "model_hidden_dim": config.hidden_dim,
        "model_num_heads": config.num_heads,
        "model_num_layers": config.num_layers,
    },
    "prs": [
        {
            "pr_id": f"{pr['repo_short']}#{pr['pr_number']}",
            "repo": pr["repo_short"],
            "pr_number": pr["pr_number"],
            "pr_title": pr["pr_title"],
            "pr_url": pr["pr_url"],
        }
        for pr in all_prs
    ],
    "llm_only_results": llm_only_results,
    "augmented_results": augmented_results,
    "structural_contexts": {
        pr_id: {
            "num_seed_nodes": ctx.get("num_seed_nodes", 0),
            "subgraph_size": ctx.get("subgraph_size", 0),
            "num_attention_edges": len(ctx.get("top_attention_edges", [])),
            "num_cross_repo": len(ctx.get("cross_repo_edges", [])),
            "num_cross_module": len(ctx.get("cross_module_edges", [])),
            "num_high_sim_neighbors": len(ctx.get("high_similarity_neighbors", [])),
            "node_profiles": ctx.get("node_profiles", []),
        }
        for pr_id, ctx in structural_contexts.items()
        if "error" not in ctx
    },
}

results_path = OUTPUT_DIR / "05_full_experiment_results.json"
with open(results_path, "w") as f:
    json.dump(full_results, f, indent=2, default=str)

# Persist to Drive
DRIVE_EXPERIMENT = DRIVE_DIR / "experiment_outputs"

DRIVE_EXPERIMENT.mkdir(exist_ok=True)
print(f"  File size: {results_path.stat().st_size / 1024:.1f} KB")

shutil.copy2(results_path, DRIVE_EXPERIMENT / results_path.name)
print(f"✓ Saved to {results_path} (+ Drive)")

  File size: 428.8 KB
✓ Saved to /content/rgat_project/experiment_outputs/05_full_experiment_results.json (+ Drive)


In [45]:
# ── Export comparison CSV for easy analysis ───────────────────────────
if len(df_comparison) > 0:
    csv_path = OUTPUT_DIR / "05_comparison_table.csv"
    df_comparison.to_csv(csv_path, index=False)
    DRIVE_EXPERIMENT = DRIVE_DIR / "experiment_outputs"

    DRIVE_EXPERIMENT.mkdir(exist_ok=True)    
    print(f"✓ Comparison table saved to {csv_path} (+ Drive)")
    shutil.copy2(csv_path, DRIVE_EXPERIMENT / csv_path.name)

✓ Comparison table saved to /content/rgat_project/experiment_outputs/05_comparison_table.csv (+ Drive)


In [46]:
# ── Generate formatted output for LLM-as-judge evaluation ────────────
# This creates a document you can paste into Copilot for manual judging

judge_doc_parts = []
judge_doc_parts.append("# Change Risk Assessment — LLM-as-Judge Evaluation\n")
judge_doc_parts.append(
    "For each PR below, two risk assessments are provided:\n"
    "- **Assessment A**: Uses only the PR diff and local code context\n"
    "- **Assessment B**: Uses the PR diff PLUS structural intelligence from an RGAT model\n\n"
    "For each PR, evaluate which assessment is more **complete** and **accurate**, "
    "considering: risk identification, propagation analysis, and architectural awareness.\n\n"
    "Rate each assessment 1-5 on: Completeness, Accuracy, Specificity.\n"
    "Then indicate which is better overall (A, B, or Tie).\n"
)

for aug in augmented_results:
    pr_id = aug["pr_id"]
    llm_only = llm_only_by_pr.get(pr_id, {})
    aug_assess = aug["assessment"]

    if "error" in llm_only or "error" in aug_assess:
        continue

    # Find matching PR data
    pr_data = next((p for p in all_prs if f"{p['repo_short']}#{p['pr_number']}" == pr_id), None)
    if not pr_data:
        continue

    judge_doc_parts.append(f"\n{'='*70}")
    judge_doc_parts.append(f"## PR: {pr_id} — {aug['pr_title']}")
    judge_doc_parts.append(f"URL: {pr_data['pr_url']}")
    judge_doc_parts.append(f"Files changed: {pr_data['changed_files_count']} (+{pr_data['additions']}/−{pr_data['deletions']})")

    judge_doc_parts.append(f"\n### Assessment A (LLM-Only)")
    judge_doc_parts.append(f"Risk Score: {llm_only.get('risk_score')} ({llm_only.get('risk_level')}) [S={llm_only.get('severity')}, P={llm_only.get('probability')}]")
    judge_doc_parts.append(f"Summary: {llm_only.get('summary', 'N/A')}")
    judge_doc_parts.append(f"Key Risks: {json.dumps(llm_only.get('key_risks', []))}")
    judge_doc_parts.append(f"Affected Areas: {json.dumps(llm_only.get('affected_areas', []))}")
    judge_doc_parts.append(f"Severity Reasoning: {llm_only.get('severity_reasoning', 'N/A')}")
    judge_doc_parts.append(f"Probability Reasoning: {llm_only.get('probability_reasoning', 'N/A')}")

    judge_doc_parts.append(f"\n### Assessment B (LLM+RGAT)")
    judge_doc_parts.append(f"Risk Score: {aug_assess.get('risk_score')} ({aug_assess.get('risk_level')}) [S={aug_assess.get('severity')}, P={aug_assess.get('probability')}]")
    judge_doc_parts.append(f"Summary: {aug_assess.get('summary', 'N/A')}")
    judge_doc_parts.append(f"Key Risks: {json.dumps(aug_assess.get('key_risks', []))}")
    judge_doc_parts.append(f"Affected Areas: {json.dumps(aug_assess.get('affected_areas', []))}")
    judge_doc_parts.append(f"Severity Reasoning: {aug_assess.get('severity_reasoning', 'N/A')}")
    judge_doc_parts.append(f"Probability Reasoning: {aug_assess.get('probability_reasoning', 'N/A')}")
    if "structural_insights" in aug_assess:
        judge_doc_parts.append(f"Structural Insights: {json.dumps(aug_assess['structural_insights'])}")

    judge_doc_parts.append(f"\n### Judge Response")
    judge_doc_parts.append("Completeness: A=? B=?")
    judge_doc_parts.append("Accuracy: A=? B=?")
    judge_doc_parts.append("Specificity: A=? B=?")
    judge_doc_parts.append("Overall Winner: A / B / Tie")
    judge_doc_parts.append("Reasoning: ...")

judge_doc = "\n".join(judge_doc_parts)

judge_path = OUTPUT_DIR / "05_judge_evaluation_template.md"
with open(judge_path, "w") as f:
    f.write(judge_doc)

DRIVE_EXPERIMENT = DRIVE_DIR / "experiment_outputs"
DRIVE_EXPERIMENT.mkdir(exist_ok=True)
shutil.copy2(judge_path, DRIVE_EXPERIMENT / judge_path.name)

print(f"\nYou can now open this file and use Copilot as the LLM judge.")

print(f"✓ Judge evaluation template saved to {judge_path} (+ Drive)")
print(f"  {len(judge_doc):,} chars, covering {len([a for a in augmented_results if 'error' not in a['assessment']])} PRs")


You can now open this file and use Copilot as the LLM judge.
✓ Judge evaluation template saved to /content/rgat_project/experiment_outputs/05_judge_evaluation_template.md (+ Drive)
  177,457 chars, covering 55 PRs


---

## Summary

This notebook completed the change risk experiment:

1. **Loaded the trained RGAT model** and computed node embeddings + attention weights
2. **Extracted structural context** for each PR: centrality profiles, attention-weighted dependency paths, cross-module/cross-repo connections, and embedding similarities
3. **Re-prompted GPT-4o** with diff + structural context (LLM+RGAT condition)
4. **Generated side-by-side comparison** of LLM-only vs. LLM+RGAT assessments
5. **Exported judge evaluation template** for manual LLM-as-judge review

### Output Files
- `experiment_outputs/05_full_experiment_results.json` — Complete results
- `experiment_outputs/05_comparison_table.csv` — Tabular comparison
- `experiment_outputs/05_judge_evaluation_template.md` — Template for LLM-as-judge